#**Part 3 — Machine Learning: predict no-shows and choose the model honestly**

##Task 3.1 — Features, split and baseline

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
# Load data without sorting or resetting the row index

df = pd.read_csv("/content/appointments_clean.csv")

#Keep IDs for later; do not use them as model features
appointment_ids = df['appointment_id'].copy

X = df[
    ['age',
     'distance_km',
     'fee_inr',
     'lead_days',
     'prior_no_shows',
     'reminder_sent',
     'booking_channel',
     'visit_type'
     ]
].copy()


#Convert reminder from Yes/No to 1/0

X["reminder_sent"] = X['reminder_sent'].map({"Yes":1,"No":0})

#One-hot encode the two categorical features
X = pd.get_dummies(
    X,
    columns=["booking_channel", "visit_type"],
    dtype= int
)

#Target
y = df["no_show"].astype(int)

#Split with the required settings
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size = 0.2, random_state = 42)

#Baseline: always predict Attended (no-show= 0)

baseline_accuracy = (y_test == 0).mean()

print("\nFeature columns:", X.columns.tolist())
print("\nNumber of features:", X.shape[1])
print("\nTraining rows:", len(X_train))
print("\nTest rows:", len(X_test))
print("\nNo-shows in test set:", int(y_test.sum()))
print(f"\nAlways-Attended baseline accuracy: {baseline_accuracy:.4f}")



Feature columns: ['age', 'distance_km', 'fee_inr', 'lead_days', 'prior_no_shows', 'reminder_sent', 'booking_channel_App', 'booking_channel_Phone', 'booking_channel_Walk-In', 'visit_type_Consultation', 'visit_type_Follow-up']

Number of features: 11

Training rows: 792

Test rows: 198

No-shows in test set: 49

Always-Attended baseline accuracy: 0.7525


##Task 3.2 — Logistic regression

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score

)

#Fit the scaler on training data only
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

#Fit logistic regression
lr_model = LogisticRegression(max_iter= 1000)
lr_model.fit(X_train_s, y_train)

#Predict classes and probabilities on the test set
y_pred = lr_model.predict(X_test_s)
y_prob = lr_model.predict_proba(X_test_s)[:, 1]

#Test metrics
print("\nAccuracy:", round(accuracy_score(y_test, y_pred),3))
print("\nConfusion matrix [[TN, FP],[FN, TP]]:")
print(confusion_matrix(y_test, y_pred, labels=[0, 1]))
print("\nPrecision:",round(precision_score(y_test,y_pred, zero_division=0), 3))
print("\nRecall:", round(recall_score(y_test, y_pred, zero_division=0), 3))
print("\nF1:", round(f1_score(y_test, y_pred, zero_division=0),3))
print("\nROC-AUC:", round(roc_auc_score(y_test, y_prob), 3))

#Coefficients, in the same order as X's feature columns
coefficients_table = pd.DataFrame({
    "feature": X.columns,
    "coefficient": lr_model.coef_[0]
})

print("\nCoefficients:")
print(coefficients_table.to_string(index=False))

#Reminder effect
reminder_index = X.columns.get_loc("reminder_sent")
reminder_coefficient = lr_model.coef_[0][reminder_index]
reminder_scale = scaler.scale_[reminder_index]

odds_multiplier_per_standard_deviation = np.exp(reminder_coefficient)
odds_multiplier_no_to_yes = np.exp(reminder_coefficient/ reminder_scale)

print("\nReminder Coefficient:", round(reminder_coefficient, 4))
print("\nReminder Standard Deviation:", round(reminder_scale, 4))
print("\nOdds multiplier per standard deviation:", round(odds_multiplier_per_standard_deviation, 3))
print("\nOdds multiplier for No -> Yes:", round(odds_multiplier_no_to_yes, 3))



Accuracy: 0.788

Confusion matrix [[TN, FP],[FN, TP]]:
[[139  10]
 [ 32  17]]

Precision: 0.63

Recall: 0.347

F1: 0.447

ROC-AUC: 0.789

Coefficients:
                feature  coefficient
                    age    -0.253880
            distance_km     0.253448
                fee_inr     0.452761
              lead_days     0.614471
         prior_no_shows     0.649612
          reminder_sent    -0.580669
    booking_channel_App    -0.046804
  booking_channel_Phone     0.215881
booking_channel_Walk-In    -0.186364
visit_type_Consultation     0.015658
   visit_type_Follow-up    -0.015658

Reminder Coefficient: -0.5807

Reminder Standard Deviation: 0.495

Odds multiplier per standard deviation: 0.56

Odds multiplier for No -> Yes: 0.309


In this model, holding the other features constant, having a reminder was associated with lower modeled odds of a no-show (odds multiplier: 0.309); this observational data does not show that reminders caused the reduction.

##Task 3.3 — Decision tree and overfitting

In [ ]:
from sklearn.tree import DecisionTreeClassifier

In [ ]:
#Full-depth tree
full_tree = DecisionTreeClassifier(random_state=42)
full_tree.fit(X_train, y_train)

full_train_accuracy = full_tree.score(X_train, y_train)
full_test_accuracy = full_tree.score(X_test, y_test)

#Tree limited to depth 3
depth3_tree = DecisionTreeClassifier(max_depth=3, random_state=42)
depth3_tree.fit(X_train, y_train)

depth3_train_accuracy = depth3_tree.score(X_train, y_train)
depth3_test_accuracy = depth3_tree.score(X_test, y_test)

print("\nFull tree depth:", full_tree.get_depth())
print(f"\nFull tree training accuracy: {full_train_accuracy:.3f}")
print(f"\nFull tree test accuracy: {full_test_accuracy:.3f}")
print()
print(f"\nDepth-3 tree training accuracy: {depth3_train_accuracy:.3f}")
print(f"\nDepth-3 tree test accuracy:     {depth3_test_accuracy:.3f}")


Full tree depth: 15

Full tree training accuracy: 1.000

Full tree test accuracy: 0.672


Depth-3 tree training accuracy: 0.822

Depth-3 tree test accuracy:     0.778


The full-depth tree achieved 1.00 training accuracy but about 0.67 test accuracy, a large gap that indicates overfitting. Limiting the tree to depth 3 narrowed the gap and raised test accuracy to about 0.78, suggesting better performance on unseen data.

##Task 3.4 — Random forest and a comparison table

In [ ]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

#Random forest training and test results
rf_train_accuracy = rf.score(X_train, y_train)
rf_test_pred = rf.predict(X_test)
rf_test_prob = rf.predict_proba(X_test)[:, 1]

print(f"\nRandom forest training accuracy:  {rf_train_accuracy:.3f}")
print(f"\nRandom forest test accuracy:      {accuracy_score(y_test, rf_test_pred):.3f}")
print(f"\nRandom forest test recall:        {recall_score(y_test, rf_test_pred):.3f}")
print(f"\nRandom forest test ROC-AUC:       {roc_auc_score(y_test, rf_test_prob):.3f}")

#Top three random forest feature importances
top_three = (
    pd.Series(rf.feature_importances_, index=X.columns)
    .sort_values(ascending=False)
    .head(3)
)

print("\nTop three feature importances:")
print(top_three.to_string())




Random forest training accuracy:  1.000

Random forest test accuracy:      0.747

Random forest test recall:        0.306

Random forest test ROC-AUC:       0.706

Top three feature importances:
lead_days      0.219905
age            0.196788
distance_km    0.194739


In [ ]:
#Collect test metrics for the comparison table

def test_metrics(model, X_eval):
  predictions = model.predict(X_eval)
  probabilities = model.predict_proba(X_eval)[:, 1]

  return{
      "accuracy": accuracy_score(y_test, predictions),
      "recall": recall_score(y_test, predictions),
      "ROC-AUC": roc_auc_score(y_test, probabilities)
  }

comparison = pd.DataFrame([
     {"model": "Logistic Regression", **test_metrics(lr_model, X_test_s)},
     {"model": "Depth-3 tree", **test_metrics(depth3_tree, X_test)},
     {"model": "Random forest", **test_metrics(rf, X_test)}
]).set_index("model")

print("\nTest-set comparison:")
print(comparison.round(3))


Test-set comparison:
                     accuracy  recall  ROC-AUC
model                                         
Logistic Regression     0.788   0.347    0.789
Depth-3 tree            0.778   0.224    0.700
Random forest           0.747   0.306    0.706


In this test-set comparison, logistic regression looks strongest: it has the highest accuracy (0.788), recall (0.347), and ROC-AUC (0.789). The formal model choice will be based on training-set cross-validation in Task 3.5, not this test-set comparison.

##Task 3.5 — Tune, choose the model and the threshold, then test once

In [ ]:
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_predict

In [ ]:
# Task 3.5(a): tune a random forest using training data only

rf_grid = GridSearchCV(
    estimator = RandomForestClassifier(random_state=42),
    param_grid={
        "n_estimators": [100,200],
        "max_depth": [3,5,8]
    },
    cv=5,
    scoring="roc_auc"
)

rf_grid.fit(X_train, y_train)

print("\nTuned forest best parameters:", rf_grid.best_params_)
print(f"\nTuned forest best mean CV ROC-AUC: {rf_grid.best_score_:.3f}")



Tuned forest best parameters: {'max_depth': 3, 'n_estimators': 200}

Tuned forest best mean CV ROC-AUC: 0.767


In [ ]:
# Task 3.5(b): compare mean five-fold ROC-AUC scores

# Scaling is inside the pipeline, so each fold fits its own scaler.

lr_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000))
])

depth3_cv_model = DecisionTreeClassifier(max_depth=3, random_state=42)

rf_cv_model = RandomForestClassifier(n_estimators=100, random_state=42)

lr_cv_auc = cross_val_score(lr_pipe, X_train, y_train, cv=5, scoring="roc_auc").mean()

depth3_cv_auc = cross_val_score(depth3_cv_model, X_train, y_train, cv=5, scoring="roc_auc").mean()

rf_cv_auc = cross_val_score(rf_cv_model, X_train, y_train, cv=5, scoring="roc_auc").mean()

cv_summary = pd.DataFrame({
    "model": [
        "Logistic regression",
        "Depth-3 tree",
        "Random forest",
        "Tuned random forest"
    ],
    "mean_cv_roc_auc": [
        lr_cv_auc,
        depth3_cv_auc,
        rf_cv_auc,
        rf_grid.best_score_
    ]
}).sort_values("mean_cv_roc_auc", ascending=False)

print("\nTraining-set cross-validation comparison:")
print(cv_summary.to_string(index=False, formatters={
    "mean_cv_roc_auc": "{:.3f}".format
}))


print("\nHighest mean CV ROC-AUC:", cv_summary.iloc[0]["model"])


Training-set cross-validation comparison:
              model mean_cv_roc_auc
Logistic regression           0.774
Tuned random forest           0.767
      Random forest           0.747
       Depth-3 tree           0.678

Highest mean CV ROC-AUC: Logistic regression


In [ ]:
# Task 3.5(c): get out-of-fold probabilities from the logistic pipeline on the training set.

# Recreate the logistic regression pipeline
lr_pipe = Pipeline([("scaler", StandardScaler()), ("model", LogisticRegression(max_iter=1000))])

#Out of fold probabilities for training rows

oof_probabilities = cross_val_predict(
    lr_pipe,
    X_train,
    y_train,
    cv=5,
    method="predict_proba"
)[:, 1]

#Compare the two required thresholds
threshold_results = []

for threshold in [0.5, 0.35]:
  oof_predictions = (oof_probabilities >= threshold).astype(int)

  threshold_results.append({
      "threshold": threshold,
      "precision": precision_score(y_train, oof_predictions, zero_division=0),
      "recall": recall_score(y_train, oof_predictions, zero_division=0),
      "f1": f1_score(y_train, oof_predictions, zero_division=0),
      "number_flagged": int(oof_predictions.sum())
  })

threshold_table = pd.DataFrame(threshold_results)

print(threshold_table.to_string(
    index=False,
    formatters={
        "precision":"{:.3f}".format,
        "recall":"{:.3f}".format,
        "f1":"{:.3f}".format
    }
))

 threshold precision recall    f1  number_flagged
      0.50     0.575  0.249 0.347              73
      0.35     0.513  0.467 0.489             154


I chose threshold 0.35 because it had the higher F1 in the out-of-fold comparison. Recall matters because a missed warning can leave a doctor’s slot empty, while an extra flagged patient mainly adds about a minute of call time.

In [ ]:
#Task 3.5(d) Fit the winning pipeline on all training rows
chosen_threshold = 0.35

lr_pipe.fit(X_train, y_train)

#Predict no-show probabilities for the test rows

test_risk_prob = lr_pipe.predict_proba(X_test)[:, 1]

#Apply the threshold chosen using training out of fold results
test_flags = (test_risk_prob >= chosen_threshold).astype(int)

#Evaluate once on the test set
test_precision = precision_score(y_test, test_flags, zero_division=0)
test_recall = recall_score(y_test, test_flags, zero_division=0)
test_roc_auc = roc_auc_score(y_test, test_risk_prob)
number_flagged = int(test_flags.sum())

print("Chosen threshold:", chosen_threshold)
print(f"Test precision:   {test_precision:.3f}")
print(f"Test recall:      {test_recall:.3f}")
print(f"Test rows flagged:",number_flagged)
print(f"Test ROC-AUC:     {test_roc_auc:.3f}")


Chosen threshold: 0.35
Test precision:   0.571
Test recall:      0.571
Test rows flagged: 49
Test ROC-AUC:     0.789


In [ ]:
#Task 3.5(e) Save test_risk_scores.csv

#Match each test row to its appointments ID using the preserved index
clean_df = pd.read_csv("appointments_clean.csv")
appointment_ids = clean_df["appointment_id"].copy()

test_risk_scores = pd.DataFrame({
    "appointment_id": appointment_ids.loc[X_test.index].to_numpy(),
    "no_show": y_test.to_numpy(),
    "risk_prob": test_risk_prob,
    "flag": test_flags
})

test_risk_scores.to_csv("test_risk_scores.csv", index=False)

print("Rows saved       :", len(test_risk_scores))
print("\nColumns        :",test_risk_scores.columns.tolist())
print("\nRows flagged   :",int(test_risk_scores["flag"].sum()))
print(test_risk_scores.head())

Rows saved       : 198

Columns        : ['appointment_id', 'no_show', 'risk_prob', 'flag']

Rows flagged   : 49
  appointment_id  no_show  risk_prob  flag
0          A0217        0   0.021565     0
1          A0336        0   0.150917     0
2          A0508        0   0.518692     1
3          A0314        0   0.191300     0
4          A0089        0   0.167718     0
